# GAT + MCTS router: train on Colab

This notebook sets everything up, checks it, trains, scores the result on the hard tier, and then **shuts the runtime down so it stops using compute units**.

**How to use it**
1. **Runtime → Change runtime type → T4 GPU** (any GPU works; more CPU cores make self-play faster).
2. Fill in the **Settings** cell below (at least `GAT_REPO_URL`).
3. **Runtime → Run all**. Approve the Google Drive pop-up at the start; after that it runs unattended.

**What happens, in order**
| step | time (roughly) |
|---|---|
| setup: Drive, clone the challenge toolkit + your repo, install PyTorch Geometric | 1–2 min |
| self-test: every step at tiny scale, so a problem shows up before the budget is spent | 3–5 min |
| training data: 150 generated instances, saved to Drive (only the first time) | 15–25 min |
| training: warm start, then MCTS self-play until `TRAIN_HOURS` is used up | `TRAIN_HOURS` |
| evaluation: route + score all 9 hard-tier cases, write a leaderboard-format submission | 20–40 min |
| report + automatic shutdown | – |

**Budget safety**
* If any step fails, the error is printed and the runtime is shut down (when `AUTO_SHUTDOWN = True`), so a failure doesn't sit there using units.
* `TRAIN_HOURS` is the total over all sessions of the same `RUN_NAME`: if Colab disconnects, just **Run all** again. The data is reused and training resumes from the last checkpoint without going over budget.
* Everything is saved in Google Drive under `MyDrive/gat_mcts_work/`.

## Settings

In [ ]:
# ---- edit these ----------------------------------------------------------------
GAT_REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"  # your repo containing the gat_mcts/ folder
GAT_REPO_BRANCH = "main"
GAT_REPO_SUBDIR = ""            # folder inside the repo that contains gat_mcts/ ("" = repo root)
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # private repo only: add a Colab secret with this name
                                      # (key icon in the left bar) holding a GitHub token

RUN_NAME = "run1"               # checkpoints/logs go to MyDrive/gat_mcts_work/<RUN_NAME>
TRAIN_HOURS = 3.0               # total training time for this run, across sessions
N_TRAIN_INSTANCES = 150         # size of the generated training pool
RUN_SELFTEST = True             # 3-5 min check before training; leave on
EVALUATE = True                 # score all hard-tier cases after training
EVAL_SIMS = 32                  # MCTS simulations per move when scoring
AUTO_SHUTDOWN = True            # disconnect the runtime at the end or on any error
# ---------------------------------------------------------------------------------

CHALLENGE_URL = "https://github.com/partcleda/eda-3d-routing-challenge.git"
CHALLENGE_COMMIT = "49b6443d9b599b77e09432c379ea7f422bdf4ffc"  # the version gat_mcts was tested on
WORK = "/content/drive/MyDrive/gat_mcts_work"
CHALLENGE_DIR = "/content/eda-3d-routing-challenge"
REPO_DIR = "/content/gat_repo"
DATA_LOCAL = "/content/gat_data"

import os, sys, json, shutil, subprocess, traceback

if "YOUR_USER" in GAT_REPO_URL:
    raise ValueError("Set GAT_REPO_URL in the Settings cell to the repo that holds gat_mcts/, then Run all again.")

ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0", M3D_CHALLENGE_ROOT=CHALLENGE_DIR)

def sh(cmd, cwd=None, show=None):
    # Run a command, streaming its output into this cell; raise if it fails.
    print("$", show or cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            if "UserWarning" in line or line.lstrip().startswith("warnings.warn"):
                continue
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        p.terminate()
        raise
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {show or cmd}")

def stop_runtime(reason):
    print(f"\n=== {reason} ===", flush=True)
    if not AUTO_SHUTDOWN:
        print("AUTO_SHUTDOWN is False: the runtime keeps running (and using units) until you disconnect it.")
        return
    print("Saving Google Drive and disconnecting the runtime so it stops using compute units...", flush=True)
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()
    except Exception as exc:
        print("Drive flush:", exc)
    runtime.unassign()

class guard:
    # Wrap a step: on an error (not a manual stop) print it and shut the runtime down.
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        return self
    def __exit__(self, et, ev, tb):
        if et is not None and not issubclass(et, KeyboardInterrupt):
            traceback.print_exception(et, ev, tb)
            stop_runtime(f"step '{self.name}' failed: see the error above")
        return False

print("settings ok")

## 1. Setup

In [ ]:
with guard("setup"):
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(WORK, exist_ok=True)

    # the challenge toolkit + benchmarks, pinned to the tested version
    if not os.path.isdir(os.path.join(CHALLENGE_DIR, ".git")):
        sh(f"git clone -q {CHALLENGE_URL} {CHALLENGE_DIR}")
    sh(f"git -C {CHALLENGE_DIR} checkout -q {CHALLENGE_COMMIT}")

    # your repo (always a fresh copy, so the latest pushed code is used)
    token = None
    try:
        from google.colab import userdata
        token = userdata.get(GITHUB_TOKEN_SECRET)
    except Exception:
        pass  # no secret: the repo must be public
    url = GAT_REPO_URL.replace("https://", f"https://{token}@") if token else GAT_REPO_URL
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    try:
        sh(f"git clone -q --depth 1 --branch {GAT_REPO_BRANCH} {url} {REPO_DIR}",
           show=f"git clone --branch {GAT_REPO_BRANCH} {GAT_REPO_URL} (token: {'yes' if token else 'no'})")
    except RuntimeError:
        raise RuntimeError("Could not clone your repo. Check GAT_REPO_URL / GAT_REPO_BRANCH; "
                           "if the repo is private, add the GitHub token as a Colab secret "
                           f"named {GITHUB_TOKEN_SECRET} and allow this notebook to use it.")
    CODE_DIR = os.path.join(REPO_DIR, GAT_REPO_SUBDIR)
    if not os.path.isfile(os.path.join(CODE_DIR, "gat_mcts", "train.py")):
        raise RuntimeError(f"gat_mcts/train.py not found under {CODE_DIR}: set GAT_REPO_SUBDIR "
                           "to the folder that contains gat_mcts/")
    ENV["PYTHONPATH"] = CODE_DIR + os.pathsep + ENV.get("PYTHONPATH", "")

    sh(f"{sys.executable} -m pip -q install torch_geometric==2.8.0.post1")
    import torch
    WORKERS = os.cpu_count() or 1
    GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    print(f"\nGPU: {GPU}   CPU cores: {WORKERS}   torch {torch.__version__}")

if GPU is None:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again "
                       "(training on CPU would burn the budget ~50x slower).")

## 2. Self-test
Runs data generation, training, resume and evaluation at tiny scale (~3–5 min). If anything is wrong with the environment it fails here, not hours into training.

In [ ]:
with guard("self-test"):
    if RUN_SELFTEST:
        sh(f"{sys.executable} -m gat_mcts.selftest --workers {WORKERS}", cwd=CODE_DIR)
    else:
        print("skipped")

## 3. Training data
Generated once and kept in Drive (resumable if interrupted), then copied to the local disk for speed.

In [ ]:
with guard("training data"):
    DATA_DRIVE = f"{WORK}/data"
    index = os.path.join(DATA_DRIVE, "index.json")
    have = len(json.load(open(index))) if os.path.exists(index) else 0
    if have < 0.9 * N_TRAIN_INSTANCES:
        sh(f"{sys.executable} -m gat_mcts.gen_data --out {DATA_DRIVE} --n {N_TRAIN_INSTANCES} "
           f"--workers {WORKERS}", cwd=CODE_DIR)
    else:
        print(f"using the existing pool in Drive ({have} instances)")
    shutil.rmtree(DATA_LOCAL, ignore_errors=True)
    shutil.copytree(DATA_DRIVE, DATA_LOCAL)
    print("instances:", len(json.load(open(os.path.join(DATA_LOCAL, "index.json")))))

## 4. Train
Resumes automatically if this run already has a checkpoint. Stops when `TRAIN_HOURS` (total across sessions) is used up. Each self-play line shows the challenge score of an episode (baseline = 1.0); every 10 iterations it scores 2 hard-tier cases.

In [ ]:
with guard("training"):
    RUN_DIR = f"{WORK}/{RUN_NAME}"
    os.makedirs(RUN_DIR, exist_ok=True)
    latest = os.path.join(RUN_DIR, "latest.pt")
    resume = f"--resume {latest}" if os.path.exists(latest) else ""
    print("resuming from", latest) if resume else print("starting a new run")
    sh(f"{sys.executable} -m gat_mcts.train --data {DATA_LOCAL} --out {RUN_DIR} {resume} "
       f"--max-hours {TRAIN_HOURS} --workers {WORKERS} --eval-every 10 --eval-cases 2", cwd=CODE_DIR)

## 5. Score the hard tier
Routes all 9 hard-tier cases with the trained network and writes a leaderboard-format submission (`case_XX.sol.json`, `runtime.json`, `meta.json`) to Drive, scored by the challenge's own scorer.

In [ ]:
with guard("evaluation"):
    SUB_DIR = f"{WORK}/submissions/hard/{RUN_NAME}"
    if EVALUATE:
        sh(f"{sys.executable} -m gat_mcts.evaluate --tier hard --ckpt {RUN_DIR}/latest.pt "
           f"--sims {EVAL_SIMS} --cleanup-seconds 60 --workers {WORKERS} --out {SUB_DIR}", cwd=CODE_DIR)
    else:
        print("skipped")

## 6. Report, then shut down

In [ ]:
with guard("report"):
    import matplotlib.pyplot as plt
    rows = [json.loads(l) for l in open(os.path.join(RUN_DIR, "log.jsonl"))]
    its = [r for r in rows if r.get("kind") == "iter"]
    if its:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot([r["iter"] for r in its], [r["mean_score"] for r in its], marker=".")
        ax[0].axhline(1.0, color="gray", lw=1, ls="--")
        ax[0].set_title("self-play score (baseline = 1.0)"); ax[0].set_xlabel("iteration")
        for k in ("policy", "value", "legal"):
            ax[1].plot([r["iter"] for r in its], [r[k] for r in its], label=k)
        ax[1].set_yscale("log"); ax[1].set_title("losses"); ax[1].legend(); ax[1].set_xlabel("iteration")
        plt.show()
    for r in rows:
        if r.get("kind") == "eval":
            print(f"iter {r['iter']}: " + ", ".join(
                f"{c['case']} search {c['search']:.4f} final {c['final']:.4f}" for c in r["cases"]))
    print(f"\ncheckpoints + log: {RUN_DIR}")
    if EVALUATE:
        print(f"submission:        {SUB_DIR}")

stop_runtime("All done")